In [32]:
import cv2
import shutil
import os
from pathlib import Path

# ==========================================
# 1. CONFIGURACIÓN DEL LABORATORIO
# ==========================================
# Cambia este ID por el video que quieras analizar
VIDEO_ID = "ebngmAqbe-k"

# Rutas absolutas basadas en tu estructura
PROJECT_ROOT = Path('/home/stemjara/Projects/AWS-Architecture')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
LAB_DIR = PROJECT_ROOT / 'whiteboard_selection_lab'

# Nuevas carpetas de destino para las pruebas
FRAMES_NEW_DIR = LAB_DIR / 'frames_new' / VIDEO_ID
TRANSCRIPTIONS_DIR = LAB_DIR / 'transcriptions'

def preparar_entorno_prueba(video_id):
    """Crea directorios, copia transcripciones y extrae frames con velocidad dinámica."""
    print(f"🚀 Iniciando preparación para el video: {video_id}")

    # 2. CREACIÓN DE CARPETAS
    FRAMES_NEW_DIR.mkdir(parents=True, exist_ok=True)
    TRANSCRIPTIONS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✅ Carpetas creadas:\n  - {FRAMES_NEW_DIR}\n  - {TRANSCRIPTIONS_DIR}")

    # Limpiar frames anteriores de esta carpeta si existen (para evitar mezclas)
    for f in FRAMES_NEW_DIR.glob('*.jpg'):
        f.unlink()

    # 3. COPIAR LA TRANSCRIPCIÓN
    transcript_origen = RAW_DIR / f"{video_id}_transcript.json"
    transcript_destino = TRANSCRIPTIONS_DIR / f"{video_id}_transcript.json"

    if transcript_origen.exists():
        shutil.copy(transcript_origen, transcript_destino)
        print(f"✅ Transcripción copiada a la carpeta de pruebas.")
    else:
        print(f"⚠️ ADVERTENCIA: No se encontró la transcripción en {transcript_origen}")

    # 4. EXTRACCIÓN DE FRAMES DINÁMICA (10s al inicio, 1s al final)
    video_path = RAW_DIR / f"{video_id}.mp4"

    if not video_path.exists():
        print(f"❌ ERROR: No se encontró el video en {video_path}")
        return

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"❌ ERROR: No se pudo abrir el video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30.0 # Fallback por si OpenCV falla en leer los FPS

    # Obtener el total de frames para calcular el 80%
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    umbral_80_porciento = int(total_frames * 0.80)

    # Definir los dos intervalos
    intervalo_lento = int(fps * 10) # Cada 10 segundos
    intervalo_rapido = int(fps * 1)  # Cada 1 segundo

    frame_count = 0
    saved_count = 0

    print(f"🎞️ Extrayendo frames... (FPS: {fps:.2f}, Total frames: {total_frames})")
    print(f"   ⏱️ Del 0% al 80%: Cada 10s")
    print(f"   ⏱️ Del 80% al 100%: Cada 1s")

    while True:
        ret, frame = cap.read()
        if not ret:
            break # Fin del video

        # Determinar qué intervalo usar dependiendo de en qué parte del video estamos
        intervalo_actual = intervalo_lento if frame_count < umbral_80_porciento else intervalo_rapido

        # Guardar si cumple el intervalo correspondiente
        if frame_count % intervalo_actual == 0:
            nombre_archivo = f"{video_id}_frame_{frame_count}.jpg"
            ruta_guardado = FRAMES_NEW_DIR / nombre_archivo
            cv2.imwrite(str(ruta_guardado), frame)
            saved_count += 1

        frame_count += 1

    cap.release()
    print(f"🎯 ¡Listo! Se extrajeron {saved_count} frames en total.")
    print(f"📁 Revisa la carpeta: {FRAMES_NEW_DIR}")

# Ejecutar la función
preparar_entorno_prueba(VIDEO_ID)

🚀 Iniciando preparación para el video: ebngmAqbe-k
✅ Carpetas creadas:
  - /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/frames_new/ebngmAqbe-k
  - /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/transcriptions
✅ Transcripción copiada a la carpeta de pruebas.
🎞️ Extrayendo frames... (FPS: 23.98, Total frames: 8222)
   ⏱️ Del 0% al 80%: Cada 10s
   ⏱️ Del 80% al 100%: Cada 1s
🎯 ¡Listo! Se extrajeron 100 frames en total.
📁 Revisa la carpeta: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/frames_new/ebngmAqbe-k


In [10]:
import cv2
import numpy as np
import shutil
from pathlib import Path

# Configuración
VIDEO_ID = "CE03UMddoYU" # Probemos con el video de hace 6 años
PROJECT_ROOT = Path('/home/stemjara/Projects/AWS-Architecture')
FRAMES_DIR = PROJECT_ROOT / 'whiteboard_selection_lab' / 'frames_new' / VIDEO_ID
OUTPUT_DIR = FRAMES_DIR.parent / f"{VIDEO_ID}_FINAL_SELECTION"

def run_full_pipeline():
    OUTPUT_DIR.mkdir(exist_ok=True)

    # 1. Carga (Último 40% del video)
    all_frames = sorted(FRAMES_DIR.glob("*.jpg"), key=lambda p: int(p.stem.split("_frame_")[-1]))
    if not all_frames: return
    start_idx = int(len(all_frames) * 0.6)
    candidates = all_frames[start_idx:]

    print(f"🚀 Procesando {len(candidates)} frames (Último 40% del video)...")

    # ---------------------------------------------------------
    # PASO 1: AUTO-CALIBRACIÓN DE OSCURIDAD
    # Encontramos la imagen más oscura del lote para definir el estándar de este video.
    # ---------------------------------------------------------
    max_dark_pct = 0.0
    for fp in candidates:
        img = cv2.imread(str(fp))
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        dark_pct = np.sum(gray < 50) / gray.size
        if dark_pct > max_dark_pct:
            max_dark_pct = dark_pct

    # El umbral dinámico será el 40% de la máxima oscuridad encontrada (muy permisivo)
    dynamic_dark_threshold = max_dark_pct * 0.40
    print(f"🎚️ Oscuridad Máxima: {max_dark_pct:.1%}. Umbral Dinámico fijado en: {dynamic_dark_threshold:.1%}")

    scored_frames = []

    for fp in candidates:
        img = cv2.imread(str(fp))
        if img is None: continue

        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        # ---------------------------------------------------------
        # FILTRO 1: ELIMINACIÓN DE PANTALLAS CLARAS (Dinámico)
        # ---------------------------------------------------------
        dark_px_pct = np.sum(gray < 50) / (h * w)
        if dark_px_pct < dynamic_dark_threshold:
            continue # Descartado por ser más claro que la pizarra promedio

        # ---------------------------------------------------------
        # FILTRO 2: ELIMINACIÓN DE OUTROS / LOGOS (Ausencia de piel)
        # ---------------------------------------------------------
        skin_mask = cv2.inRange(hsv, (0, 30, 60), (25, 170, 255))
        skin_pct = np.sum(skin_mask > 0) / skin_mask.size
        if skin_pct < 0.015:
            continue # ¡ELIMINADO! Sin humanos, no hay pizarra que valga.

        # ---------------------------------------------------------
        # SCORING DE LA PIZARRA
        # ---------------------------------------------------------
        # 1. Enmascarar íconos cuadrados de negro
        sat_mask = (hsv[:, :, 1] > 80) & (hsv[:, :, 2] > 60)
        contours, _ = cv2.findContours(sat_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        img_masked = img.copy()
        for cnt in contours:
            x, y, cw, ch = cv2.boundingRect(cnt)
            if 0.05*min(h,w) < cw < 0.15*min(h,w) and 0.85 < (cw/ch) < 1.15:
                cv2.rectangle(img_masked, (x, y), (x+cw, y+ch), (0,0,0), -1)

        # 2. Densidad de Tiza (Centro)
        roi_gray = gray[int(h*0.1):int(h*0.9), int(w*0.28):int(w*0.72)]
        edges = cv2.Canny(cv2.medianBlur(roi_gray, 7), 80, 200)
        edge_density = np.sum(edges > 0) / edges.size

        # 3. Penalización por Oclusión Humana (Centro)
        roi_skin = skin_mask[int(h*0.15):int(h*0.95), int(w*0.28):int(w*0.72)]
        occlusion = np.sum(roi_skin > 0) / roi_skin.size # Ahora de 0.0 a 1.0

        # 4. Cálculo final del Score
        score = (edge_density * 50) - (occlusion * 5)
        scored_frames.append({"path": fp, "score": score})

    # 3. Selección Final
    if not scored_frames:
        print("⚠️ Todos los frames fueron eliminados. Ninguno cumplió los requisitos.")
        return

    scored_frames.sort(key=lambda x: x['score'], reverse=True)

    # Limpiar carpeta de salida anterior
    for f in OUTPUT_DIR.glob("*.jpg"):
        f.unlink()

    print(f"\n✅ Top 10 seleccionados:")
    for i, item in enumerate(scored_frames[:10]):
        rank = i + 1
        shutil.copy(item['path'], OUTPUT_DIR / f"Rank{rank}_{item['path'].name}")
        print(f"Rank {rank}: {item['path'].name} (Score: {item['score']:.2f})")

run_full_pipeline()

🚀 Procesando 38 frames (Último 40% del video)...
🎚️ Oscuridad Máxima: 80.1%. Umbral Dinámico fijado en: 32.0%

✅ Top 10 seleccionados:
Rank 1: DnTQ3matqts_frame_19588.jpg (Score: 1.70)
Rank 2: DnTQ3matqts_frame_19293.jpg (Score: 1.70)
Rank 3: DnTQ3matqts_frame_19529.jpg (Score: 1.69)
Rank 4: DnTQ3matqts_frame_19352.jpg (Score: 1.68)
Rank 5: DnTQ3matqts_frame_19411.jpg (Score: 1.67)
Rank 6: DnTQ3matqts_frame_19470.jpg (Score: 1.66)
Rank 7: DnTQ3matqts_frame_18054.jpg (Score: 1.49)
Rank 8: DnTQ3matqts_frame_17995.jpg (Score: 1.48)
Rank 9: DnTQ3matqts_frame_17523.jpg (Score: 1.48)
Rank 10: DnTQ3matqts_frame_17818.jpg (Score: 1.48)


In [29]:
import json
import shutil
import cv2
from pathlib import Path

# ==========================================
# 1. RUTAS Y CONFIGURACIÓN
# ==========================================
VIDEO_ID = "CE03UMddoYU"

PROJECT_ROOT = Path('/home/stemjara/Projects/AWS-Architecture')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
LAB_DIR = PROJECT_ROOT / 'whiteboard_selection_lab'

# Carpetas de origen y destino
FRAMES_DIR = LAB_DIR / 'frames_new' / VIDEO_ID
TRANSCRIPT_PATH = LAB_DIR / 'transcriptions' / f"{VIDEO_ID}_transcript.json"
WHISPER_REJECT_DIR = FRAMES_DIR / f"{VIDEO_ID}_whisper"

def probar_filtro_whisper(video_id):
    print(f"🕵️ Iniciando Filtro de Tiempo (Whisper) para: {video_id}")

    # 2. CREAR CARPETA DE RECHAZADOS
    WHISPER_REJECT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📁 Carpeta de descartes creada en: {WHISPER_REJECT_DIR.name}/")

    # 3. LEER EL ÚLTIMO SEGUNDO HABLADO
    if not TRANSCRIPT_PATH.exists():
        print(f"❌ ERROR: No se encontró la transcripción en {TRANSCRIPT_PATH}")
        return

    with open(TRANSCRIPT_PATH, "r", encoding="utf-8") as f:
        segmentos = json.load(f)

    if not segmentos:
        print("❌ La transcripción está vacía.")
        return

    ultimo_segundo_hablado = segmentos[-1].get("end")
    print(f"🎙️ Última palabra detectada en el segundo: {ultimo_segundo_hablado:.2f}s")

    # 4. OBTENER LOS FPS DEL VIDEO ORIGINAL
    video_path = RAW_DIR / f"{video_id}.mp4"
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()

    # 5. FILTRAR Y MOVER
    frames = sorted(FRAMES_DIR.glob(f"{video_id}_frame_*.jpg"))
    movidos_count = 0

    for fp in frames:
        # Extraer el número de frame del nombre del archivo
        try:
            frame_num = int(fp.stem.split("_frame_")[-1])
        except ValueError:
            continue

        # Calcular en qué segundo del video ocurre este frame
        tiempo_frame_segundos = (frame_num / fps) + 1

        # Si el frame aparece DESPUÉS de que el presentador dejó de hablar...
        # (Puedes sumarle +2 o +3 a ultimo_segundo_hablado si ves que corta muy al ras)
        if tiempo_frame_segundos > ultimo_segundo_hablado:
            destino = WHISPER_REJECT_DIR / fp.name
            shutil.move(str(fp), str(destino))
            movidos_count += 1

    print(f"🎯 ¡Listo! Se movieron {movidos_count} frames a la carpeta de descartes.")
    print(f"👀 Revisa la carpeta {WHISPER_REJECT_DIR.name}/ para ver si cortó bien.")

# Ejecutar la prueba
probar_filtro_whisper(VIDEO_ID)

🕵️ Iniciando Filtro de Tiempo (Whisper) para: CE03UMddoYU
📁 Carpeta de descartes creada en: CE03UMddoYU_whisper/
🎙️ Última palabra detectada en el segundo: 413.90s
🎯 ¡Listo! Se movieron 4 frames a la carpeta de descartes.
👀 Revisa la carpeta CE03UMddoYU_whisper/ para ver si cortó bien.


In [7]:
import json
import pandas as pd
import cv2
import numpy as np
import shutil
from pathlib import Path

# ==========================================
# 1. RUTAS Y CONFIGURACIÓN
# ==========================================
VIDEO_ID = "DnTQ3matqts" # Mismo ID que estamos usando

PROJECT_ROOT = Path('/home/stemjara/Projects/AWS-Architecture')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
LAB_DIR = PROJECT_ROOT / 'whiteboard_selection_lab'
CSV_PATH = PROJECT_ROOT / 'videos.csv'

# Carpetas de origen y destino
FRAMES_DIR = LAB_DIR / 'frames_new' / VIDEO_ID
OUTROS_REJECT_DIR = FRAMES_DIR / f"{VIDEO_ID}_outros"

def get_video_age_from_csv(video_id: str) -> str:
    """Busca la edad del video usando info.json y videos.csv"""
    info_path = RAW_DIR / f"{video_id}.info.json"

    if not info_path.exists() or not CSV_PATH.exists():
        return "unknown"

    try:
        with open(info_path, "r", encoding="utf-8") as f:
            info = json.load(f)

        title = info.get("title", "").strip().lower()
        if not title:
            return "unknown"

        df = pd.read_csv(CSV_PATH)
        title_clean = "".join(title.split()).replace(":", "").replace("-", "")

        for _, row in df.iterrows():
            row_title = str(row["title"]).strip().lower()
            row_title_clean = "".join(row_title.split()).replace(":", "").replace("-", "")

            if title_clean in row_title_clean or row_title_clean in title_clean:
                return str(row["age"]).strip()
    except Exception as e:
        print(f"Error leyendo CSV: {e}")

    return "unknown"

def _is_branded_screen(img: np.ndarray) -> tuple[bool, list[str]]:
    """Lógica original de detección de Outros/Logos"""
    h, w, _ = img.shape
    reasons = []

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 1. Porcentaje de piel (Si no hay piel humana = bandera roja)
    skin_mask = cv2.inRange(hsv, (0, 40, 80), (25, 170, 255))
    skin_pct = (np.sum(skin_mask > 0) / skin_mask.size) * 100
    if skin_pct < 1.5:
        reasons.append(f"baja piel ({skin_pct:.2f}%)")

    # 2. Distribución de brillo
    col_means = np.mean(gray, axis=0)
    left_bright = np.mean(col_means[:int(w * 0.15)])
    center_bright = np.mean(col_means[int(w * 0.35):int(w * 0.65)])
    right_bright = np.mean(col_means[int(w * 0.85):])

    svc = ((left_bright + right_bright) / 2) - center_bright

    if left_bright < 32.0 and right_bright < 32.0:
        reasons.append(f"lados oscuros (L={left_bright:.1f}, R={right_bright:.1f})")
    if svc < -5.0:
        reasons.append(f"centro más brillante que lados ({svc:.1f})")

    # Es outro si tiene 2 banderas rojas y una de ellas es problema de luz
    is_outro = len(reasons) >= 2 and (svc < -5.0 or (left_bright < 32.0 and right_bright < 32.0))

    return is_outro, reasons

def probar_filtro_outros(video_id):
    print(f"🛡️ Iniciando Filtro de Outros/Logos para: {video_id}")

    # 1. LEER CSV PARA DETERMINAR UMBRAL
    age = get_video_age_from_csv(video_id)
    if age == "hace 1 año" or age == "unknown":
        dark_threshold = 75.0
    else:
        dark_threshold = 38.0

    print(f"📄 Video referenciado. Edad: '{age}'.")
    print(f"🎚️ Usando umbral de oscuridad (dark_threshold) del {dark_threshold}%")

    # 2. CREAR CARPETA DE RECHAZADOS
    OUTROS_REJECT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📁 Carpeta de descartes creada en: {OUTROS_REJECT_DIR.name}/")

    # 3. FILTRAR Y MOVER
    # Solo procesamos las imágenes que sobrevivieron al filtro de Whisper (si lo corriste antes)
    frames = sorted(FRAMES_DIR.glob("*.jpg"))
    movidos_count = 0

    print("\n🔍 Analizando imágenes...")
    for fp in frames:
        img = cv2.imread(str(fp))
        if img is None:
            continue

        h, w, c = img.shape
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Primero revisamos si la imagen cumple con el umbral de oscuridad de la pizarra
        dark_px_pct = (np.sum(gray < 45) / (h * w)) * 100

        # Si NO es lo suficientemente oscura, NO es una pizarra, la movemos
        if dark_px_pct < dark_threshold:
            destino = OUTROS_REJECT_DIR / fp.name
            shutil.move(str(fp), str(destino))
            movidos_count += 1
            print(f"  ❌ {fp.name} descartada: Muy clara ({dark_px_pct:.1f}% oscuridad < {dark_threshold}%)")
            continue

        # Si pasó la prueba de oscuridad, revisamos si es un Logo/Outro corporativo
        is_outro, reasons = _is_branded_screen(img)

        if is_outro:
            destino = OUTROS_REJECT_DIR / fp.name
            shutil.move(str(fp), str(destino))
            movidos_count += 1
            reason_str = ", ".join(reasons)
            print(f"  ❌ {fp.name} descartada: Detectada como Logo/Outro ({reason_str})")

    print(f"\n🎯 ¡Listo! Se movieron {movidos_count} frames a {OUTROS_REJECT_DIR.name}/")

# Ejecutar la prueba
probar_filtro_outros(VIDEO_ID)

🛡️ Iniciando Filtro de Outros/Logos para: DnTQ3matqts
📄 Video referenciado. Edad: 'hace 6 años'.
🎚️ Usando umbral de oscuridad (dark_threshold) del 38.0%
📁 Carpeta de descartes creada en: DnTQ3matqts_outros/

🔍 Analizando imágenes...
  ❌ DnTQ3matqts_frame_0.jpg descartada: Detectada como Logo/Outro (baja piel (0.00%), lados oscuros (L=0.0, R=0.1))
  ❌ DnTQ3matqts_frame_10183.jpg descartada: Muy clara (36.9% oscuridad < 38.0%)
  ❌ DnTQ3matqts_frame_1198.jpg descartada: Muy clara (30.0% oscuridad < 38.0%)
  ❌ DnTQ3matqts_frame_11980.jpg descartada: Muy clara (36.7% oscuridad < 38.0%)
  ❌ DnTQ3matqts_frame_12579.jpg descartada: Muy clara (37.1% oscuridad < 38.0%)
  ❌ DnTQ3matqts_frame_13178.jpg descartada: Muy clara (36.1% oscuridad < 38.0%)
  ❌ DnTQ3matqts_frame_13777.jpg descartada: Muy clara (37.8% oscuridad < 38.0%)
  ❌ DnTQ3matqts_frame_14376.jpg descartada: Muy clara (34.6% oscuridad < 38.0%)
  ❌ DnTQ3matqts_frame_14975.jpg descartada: Muy clara (36.7% oscuridad < 38.0%)
  ❌ DnTQ3ma

In [11]:
import cv2
import numpy as np
import shutil
from pathlib import Path

VIDEO_ID = "DnTQ3matqts"
PROJECT_ROOT = Path('/home/stemjara/Projects/AWS-Architecture')
FRAMES_DIR = PROJECT_ROOT / 'whiteboard_selection_lab' / 'frames_new' / VIDEO_ID
STEP1_DIR = FRAMES_DIR.parent / f"{VIDEO_ID}_step1_pizarra"

def probar_filtro_oscuridad_dinamico():
    STEP1_DIR.mkdir(parents=True, exist_ok=True)
    for f in STEP1_DIR.glob("*.jpg"): f.unlink() # Limpiar pruebas anteriores

    all_frames = sorted(FRAMES_DIR.glob("*.jpg"), key=lambda p: int(p.stem.split("_frame_")[-1]))
    if not all_frames:
        print("❌ ERROR: La carpeta está vacía. ¡Vuelve a extraer los frames!")
        return

    start_idx = int(len(all_frames) * 0.6)
    candidates = all_frames[start_idx:]
    print(f"🎬 Etapa 1: Calibración Dinámica ({len(candidates)} frames)")

    # 1. Encontrar la máxima oscuridad
    max_dark_pct = 0.0
    for fp in candidates:
        img = cv2.imread(str(fp))
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        dark_pct = np.sum(gray < 50) / gray.size
        if dark_pct > max_dark_pct: max_dark_pct = dark_pct

    dynamic_threshold = max_dark_pct * 0.40
    print(f"🎚️ Oscuridad Máxima: {max_dark_pct:.1%}. Umbral Dinámico: {dynamic_threshold:.1%}")

    # 2. Filtrar
    aprobados = 0
    for fp in candidates:
        img = cv2.imread(str(fp))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        dark_pct = np.sum(gray < 50) / gray.size

        if dark_pct >= dynamic_threshold:
            shutil.copy(str(fp), str(STEP1_DIR / fp.name))
            aprobados += 1

    print(f"✅ Sobrevivieron {aprobados} frames. Revisa {STEP1_DIR.name}/")

probar_filtro_oscuridad_dinamico()

🎬 Etapa 1: Calibración Dinámica (38 frames)
🎚️ Oscuridad Máxima: 80.1%. Umbral Dinámico: 32.0%
✅ Sobrevivieron 38 frames. Revisa DnTQ3matqts_step1_pizarra/


In [12]:
def probar_filtro_logos_humanos():
    STEP2_DIR = FRAMES_DIR.parent / f"{VIDEO_ID}_step2_humanos"
    STEP2_DIR.mkdir(parents=True, exist_ok=True)
    for f in STEP2_DIR.glob("*.jpg"): f.unlink()

    frames = sorted(STEP1_DIR.glob("*.jpg"))
    if not frames: return
    print(f"\n🕵️ Etapa 2: Filtro de Logos y Piel ({len(frames)} frames)")

    aprobados = 0
    for fp in frames:
        img = cv2.imread(str(fp))
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        skin_mask = cv2.inRange(hsv, (0, 30, 60), (25, 170, 255))
        skin_pct = np.sum(skin_mask > 0) / skin_mask.size

        # Generar debug visual
        debug_img = img.copy()
        skin_bgr = cv2.cvtColor(skin_mask, cv2.COLOR_GRAY2BGR)

        if skin_pct >= 0.015:
            # Humano detectado -> Aprobado
            shutil.copy(str(fp), str(STEP2_DIR / fp.name))
            aprobados += 1
        else:
            # Descartado: Lo guardamos solo para que veas qué descartó
            cv2.putText(debug_img, f"RECHAZADO: Piel {skin_pct:.2%} < 1.5%", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
            cv2.imwrite(str(STEP2_DIR / f"REJECT_{fp.name}"), debug_img)

    print(f"✅ Sobrevivieron {aprobados} frames. Los descartes están marcados como REJECT en {STEP2_DIR.name}/")

probar_filtro_logos_humanos()


🕵️ Etapa 2: Filtro de Logos y Piel (38 frames)
✅ Sobrevivieron 38 frames. Los descartes están marcados como REJECT en DnTQ3matqts_step2_humanos/


In [33]:
import cv2
import numpy as np
import shutil
import json
from pathlib import Path

# ==========================================
# CONFIGURACIÓN
# ==========================================
VIDEO_ID = "ebngmAqbe-k"
PROJECT_ROOT = Path('/home/stemjara/Projects/AWS-Architecture')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
FRAMES_DIR = PROJECT_ROOT / 'whiteboard_selection_lab' / 'frames_new' / VIDEO_ID
OUTPUT_DIR = FRAMES_DIR.parent / f"{VIDEO_ID}_FINAL_SELECTION"

def run_whisper_centric_pipeline():
    OUTPUT_DIR.mkdir(exist_ok=True)

    # ---------------------------------------------------------
    # 1. INTELIGENCIA DE TIEMPO (WHISPER)
    # ---------------------------------------------------------
    transcript_path = PROJECT_ROOT / 'whiteboard_selection_lab' / 'transcriptions' / f"{VIDEO_ID}_transcript.json"
    ultimo_segundo_hablado = 999999.0

    if transcript_path.exists():
        with open(transcript_path, "r", encoding="utf-8") as f:
            segmentos = json.load(f)
            if segmentos:
                ultimo_segundo_hablado = segmentos[-1].get("end")
                print(f"🎙️ Whisper: Última palabra en el segundo {ultimo_segundo_hablado:.2f}s")
    else:
        print(f"⚠️ No se encontró transcripción, no se aplicará corte de tiempo.")

    # Obtener FPS para la conversión
    video_path = RAW_DIR / f"{VIDEO_ID}.mp4"
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()

    # Cargar solo el último 60% de los frames extraídos
    all_frames = sorted(FRAMES_DIR.glob("*.jpg"), key=lambda p: int(p.stem.split("_frame_")[-1]))
    if not all_frames:
        print("❌ ERROR: No hay frames para analizar.")
        return

    start_idx = int(len(all_frames) * 0.4)
    candidates = all_frames[start_idx:]
    print(f"🚀 Procesando {len(candidates)} frames candidatos...")

    # ---------------------------------------------------------
    # 2. CALIBRACIÓN DE OSCURIDAD
    # ---------------------------------------------------------
    max_dark_pct = 0.0
    for fp in candidates:
        img = cv2.imread(str(fp))
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        dark_pct = np.sum(gray < 50) / gray.size
        if dark_pct > max_dark_pct: max_dark_pct = dark_pct

    dynamic_threshold = max_dark_pct * 0.40
    print(f"🎚️ Oscuridad Máxima: {max_dark_pct:.1%}. Umbral Mínimo: {dynamic_threshold:.1%}")

    scored_frames = []

    # ---------------------------------------------------------
    # 3. FILTRADO Y SCORING
    # ---------------------------------------------------------
    for fp in candidates:
        # A. FILTRO WHISPER (Con tu +1 de seguridad)
        try:
            frame_num = int(fp.stem.split("_frame_")[-1])
        except ValueError:
            continue

        tiempo_frame_segundos = (frame_num / fps) + 1
        if tiempo_frame_segundos > ultimo_segundo_hablado:
            continue # ¡ELIMINADO POR WHISPER! (Es el outro)

        img = cv2.imread(str(fp))
        if img is None: continue

        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        # C. SCORING DE PIZARRA
        # Enmascarar íconos cuadrados de negro para no confundir al detector de humanos
        sat_mask = (hsv[:, :, 1] > 80) & (hsv[:, :, 2] > 60)
        contours, _ = cv2.findContours(sat_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        img_masked = img.copy()
        for cnt in contours:
            x, y, cw, ch = cv2.boundingRect(cnt)
            if 0.05*min(h,w) < cw < 0.15*min(h,w) and 0.85 < (cw/ch) < 1.15:
                cv2.rectangle(img_masked, (x, y), (x+cw, y+ch), (0,0,0), -1)

        # Canny (Densidad de Tiza) en la zona segura (28% al 72% del ancho)
        roi_gray = gray[int(h*0.1):int(h*0.9), int(w*0.28):int(w*0.72)]
        edges = cv2.Canny(cv2.medianBlur(roi_gray, 7), 80, 200)
        edge_density = np.sum(edges > 0) / edges.size

        # Oclusión (Piel humana) en la zona segura
        skin_mask = cv2.inRange(cv2.cvtColor(img_masked, cv2.COLOR_BGR2HSV), (0, 30, 60), (25, 170, 255))
        roi_skin = skin_mask[int(h*0.15):int(h*0.95), int(w*0.28):int(w*0.72)]
        occlusion = np.sum(roi_skin > 0) / roi_skin.size

        # Fórmula: Premia la tiza, castiga fuerte al humano estorbando
        score = (edge_density * 50) - (occlusion * 5)
        scored_frames.append({"path": fp, "score": score})

    # ---------------------------------------------------------
    # 4. GUARDAR TOP 10
    # ---------------------------------------------------------
    if not scored_frames:
        print("⚠️ Todos los frames fueron eliminados (revisa si el corte de Whisper fue muy agresivo).")
        return

    scored_frames.sort(key=lambda x: x['score'], reverse=True)

    # Limpiar carpeta
    for f in OUTPUT_DIR.glob("*.jpg"): f.unlink()

    print(f"\n✅ Top 10 seleccionados (Outros eliminados por Whisper):")
    for i, item in enumerate(scored_frames[:10]):
        rank = i + 1
        shutil.copy(item['path'], OUTPUT_DIR / f"Rank{rank}_{item['path'].name}")
        print(f"Rank {rank}: {item['path'].name} (Score: {item['score']:.2f})")

run_whisper_centric_pipeline()

🎙️ Whisper: Última palabra en el segundo 369.38s
🚀 Procesando 60 frames candidatos...
🎚️ Oscuridad Máxima: 69.7%. Umbral Mínimo: 27.9%

✅ Top 10 seleccionados (Outros eliminados por Whisper):
Rank 1: ebngmAqbe-k_frame_7268.jpg (Score: 1.92)
Rank 2: ebngmAqbe-k_frame_7291.jpg (Score: 1.92)
Rank 3: ebngmAqbe-k_frame_7245.jpg (Score: 1.92)
Rank 4: ebngmAqbe-k_frame_7314.jpg (Score: 1.92)
Rank 5: ebngmAqbe-k_frame_7222.jpg (Score: 1.92)
Rank 6: ebngmAqbe-k_frame_7199.jpg (Score: 1.91)
Rank 7: ebngmAqbe-k_frame_7176.jpg (Score: 1.91)
Rank 8: ebngmAqbe-k_frame_7153.jpg (Score: 1.91)
Rank 9: ebngmAqbe-k_frame_7061.jpg (Score: 1.75)
Rank 10: ebngmAqbe-k_frame_6854.jpg (Score: 1.75)
